# RetinaGuard: 03 - Clinical Explainability & Pathological Attribution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harsha-Vardan/RetinaGuard/blob/main/notebooks/03_Clinical_Explainability_and_Attribution.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-RetinaGuard-blue.svg)](https://github.com/Harsha-Vardan/RetinaGuard)

---

## Notebook Overview
This notebook evaluates the **interpretability, trustworthiness, and clinical safety** of **RetinaGuard**:
1. **Attribution Comparison**: Side-by-side visualization of Baseline Grad-CAM vs. RetinaGuard Spatial Lesion Attention.
2. **Shortcut Learning Audit**: Demonstrating how standard CNNs attend to camera borders and normal optic discs, whereas RetinaGuard pinpoints true microaneurysms and exudates.
3. **Quantitative Metrics**: Computing **Attribution IoU**, **Attribution Dice**, and **Pointing Game Hit Rate**.
4. **Interactive Clinical Triage Demo**: Running single-image inference with Monte-Carlo uncertainty estimation.

### Step 1: Imports and Model Initialization

In [ ]:
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from PIL import Image

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().joinpath("src").exists() else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.baseline import DRClassifier
from src.models.retinaguard import RetinaGuardModel
from src.explainability.attribution import (
    GradCAM,
    extract_lesion_attention_map,
    compute_attribution_iou,
    compute_attribution_dice,
    compute_pointing_game_hit,
)
from src.utils.reproducibility import get_device

device = get_device()
print(f"Active device: {device}")

### Step 2: Load Best Trained Checkpoints

In [ ]:
base_path = PROJECT_ROOT / "experiments" / "baseline" / "best_model.pth"
rg_path = PROJECT_ROOT / "experiments" / "retinaguard" / "best_retinaguard.pth"

# Load Baseline model
base_model = DRClassifier(architecture="resnet18", num_classes=5)
if base_path.exists():
    st = torch.load(base_path, map_location="cpu", weights_only=False)
    base_model.load_state_dict(st.get("model_state_dict", st))
    print(f"Loaded Baseline checkpoint from: {base_path}")
base_model.to(device).eval()

# Load RetinaGuard Unified model
rg_model = RetinaGuardModel(architecture="resnet18", num_classes=5, num_domains=2)
if rg_path.exists():
    st = torch.load(rg_path, map_location="cpu", weights_only=False)
    rg_model.load_state_dict(st.get("model_state_dict", st))
    print(f"Loaded RetinaGuard checkpoint from: {rg_path}")
rg_model.to(device).eval()

### Step 3: Head-to-Head Visual Attribution Inspection
Select an IDRiD patient fundus image with verified clinical lesion masks and compare attention maps.

In [ ]:
test_manifest = PROJECT_ROOT / "data" / "manifests" / "test_manifest.csv"
df_test = pd.read_csv(test_manifest)
masked = df_test[df_test["lesion_mask_path"].notna() & (df_test["dr_grade"] >= 2)]

row = masked.iloc[0]
img_path = Path(row["image_path"])
mask_path = Path(row["lesion_mask_path"])
true_grade = int(row["dr_grade"])

raw_img = Image.open(img_path).convert("RGB")
transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
tensor = transform(raw_img).unsqueeze(0).to(device)
gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

# Baseline prediction & Grad-CAM
b_logits = base_model(tensor)
b_pred = int(torch.argmax(b_logits, dim=1))
grad_cam = GradCAM(base_model, base_model.backbone.layer4)
b_cam = grad_cam.generate(tensor, target_class=b_pred)

# RetinaGuard prediction & Spatial Lesion Attention
with torch.no_grad():
    rg_out = rg_model(tensor)
    rg_pred = int(torch.argmax(rg_out["disease_logits"], dim=1))
    rg_attn = extract_lesion_attention_map(rg_model, tensor, target_size=(256, 256))

# Compute metrics
b_iou = compute_attribution_iou(b_cam, gt_mask, dilation_radius=6)
rg_iou = compute_attribution_iou(rg_attn, gt_mask, dilation_radius=6)
b_hit = compute_pointing_game_hit(b_cam, gt_mask, tolerance_radius=6)
rg_hit = compute_pointing_game_hit(rg_attn, gt_mask, tolerance_radius=6)

print(f"True Grade: {true_grade} | Baseline Pred: {b_pred} | RetinaGuard Pred: {rg_pred}")
print(f"Baseline:    Attribution IoU: {b_iou:.4f} | Pointing Game: {'HIT' if b_hit else 'MISS'}")
print(f"RetinaGuard: Attribution IoU: {rg_iou:.4f} | Pointing Game: {'HIT' if rg_hit else 'MISS'}")

### Step 4: Multi-Panel Visual Explanation Comparison

In [ ]:
img_np = np.array(raw_img.resize((256, 256)))
gt_mask_res = cv2.resize((gt_mask > 0).astype(np.uint8), (256, 256), interpolation=cv2.INTER_AREA)

b_overlay = cv2.addWeighted(img_np, 0.6, cv2.applyColorMap((b_cam * 255).astype(np.uint8), cv2.COLORMAP_JET), 0.4, 0)
rg_overlay = cv2.addWeighted(img_np, 0.6, cv2.applyColorMap((rg_attn * 255).astype(np.uint8), cv2.COLORMAP_JET), 0.4, 0)

fig, axes = plt.subplots(1, 4, figsize=(18, 5), dpi=140)
axes[0].imshow(img_np)
axes[0].set_title(f"(a) Fundus Image\nTrue Grade: {true_grade}", fontsize=11)
axes[0].axis("off")

axes[1].imshow(gt_mask_res, cmap="hot")
axes[1].set_title("(b) Clinical Lesion Mask\n(Ground Truth)", fontsize=11)
axes[1].axis("off")

axes[2].imshow(b_overlay)
axes[2].set_title(f"(c) Baseline Grad-CAM\nPred: {b_pred} | IoU: {b_iou:.3f}", fontsize=11)
axes[2].axis("off")

axes[3].imshow(rg_overlay)
axes[3].set_title(f"(d) RetinaGuard Lesion Attn\nPred: {rg_pred} | IoU: {rg_iou:.3f}", fontsize=11, fontweight="bold", color="green")
axes[3].axis("off")

plt.suptitle("Pathological Attribution Alignment: Baseline vs. RetinaGuard", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Step 5: Benchmark Evaluation Summary Report
Display the quantitative benchmark summary across the entire 111-image held-out test cohort.

In [ ]:
rep_path = PROJECT_ROOT / "outputs" / "reports" / "benchmark_comparison.md"
if rep_path.exists():
    from IPython.display import Markdown
    display(Markdown(rep_path.read_text(encoding="utf-8")))
else:
    print("Benchmark report not found. Run benchmark_comparison.py first.")